# August 2026 verification · **06 — Gold: pathways are not people**

**The claim under test:** "7.37 million pathways against 6.24 million
patients means roughly 1.1 million people on more than one pathway."

What I already know from the gate:

* **7,370,100 pathways** is rebuilt exactly, on the headline basis (reported + carry-forward estimates).
* **6.24m patients is not a count.** NHS England takes the ratio of unique NHS numbers to pathways in the Waiting List Minimum Data Set (WLMDS, a separate weekly patient-level collection) for the week closest to month end, and multiplies it by the monthly pathway total. The gate confirmed that arithmetic; it can't confirm the ratio.

So this notebook does four things:

1. sets out the ratio, patients and **surplus pathways** (pathways − patients) for all 17 months;
2. turns "1.1m people on more than one pathway" into what the arithmetic actually supports: an **upper bound**;
3. separates how much of any movement in "patients" is really about people (the ratio) and how much is just pathways;
4. checks the two collections against each other, because the ratio comes from one and the pathways from the other.

Outputs: `data/gold/06_pathways_patients/`.

In [1]:
import sys
import warnings
from pathlib import Path

import openpyxl
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
from _common import BRONZE_DIR, GOLD_DIR  # noqa: E402

OUT = GOLD_DIR / "06_pathways_patients"
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 240)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
LAST, PREV = "2026-08", "2026-07"
# The WLMDS workbook uses an Excel conditional-formatting extension openpyxl
# can't read. It only affects styling, never values, so I silence that one warning.
warnings.filterwarnings("ignore", message="Conditional Formatting extension")

## Step 0 — The gate must have passed

In [2]:
gate = pd.read_csv(GOLD_DIR / "00_reconciliation" / "reconciliation.csv")
assert gate["pass"].all(), f"reconciliation gate has {(~gate['pass']).sum()} failing checks: run 04 first"
headline = pd.read_csv(GOLD_DIR / "00_reconciliation" / "headline_rebuilt.csv", index_col="period")
periods = list(headline.index)
print(f"gate: {len(gate)} of {len(gate)} checks passed; {len(periods)} months, {periods[0]} to {periods[-1]}")

gate: 51 of 51 checks passed; 17 months, 2025-04 to 2026-08


## Step 1 — Ratio and patients, as NHS England publishes them

Both live in the Overview time series under the group header "Unique
Patients". I key every column on group + header, as in notebook 05.

In [3]:
ws = openpyxl.load_workbook(BRONZE_DIR / "nhse_rtt_published_2026-08" /
                            "RTT-Overview-Timeseries-Including-Estimates-for-Missing-Trusts-Aug26-XLS-139K-vjgRdG.xlsx",
                            read_only=True, data_only=True).worksheets[0]
orows = list(ws.iter_rows(values_only=True))
group, keys = None, []
for g_, h_ in zip(orows[10], orows[11]):
    group = str(g_).strip() if g_ else group
    keys.append(f"{group} | {str(h_).strip()}" if h_ else None)
ratio_col = next(i for i, k in enumerate(keys) if k and k.startswith("Unique Patients | Ratio"))
pat_col = next(i for i, k in enumerate(keys) if k and k.startswith("Unique Patients | Estimated number of unique patients"))
up = pd.DataFrame([{"period": r[2].strftime("%Y-%m"), "ratio": r[ratio_col], "patients": r[pat_col]}
                   for r in orows[12:] if hasattr(r[2], "strftime")]).set_index("period").loc[periods].astype(float)

pp = headline[["total_est", "total"]].rename(columns={"total_est": "pathways", "total": "pathways_reported"}).join(up)
# NHS England's own arithmetic, every month: patients = ratio x pathways incl. estimates.
gap = (pp.patients - pp.ratio * pp.pathways).abs()
assert (gap < 1).all(), f"patients != ratio x pathways in {list(gap[gap >= 1].index)}"
pp["surplus_pathways"] = pp.pathways - pp.patients
pp["surplus_share"] = 1 - pp.ratio
pp["pathways_per_patient"] = 1 / pp.ratio
pp.to_csv(OUT / "pathways_patients_by_month.csv")
print("patients = ratio × pathways (incl. estimates) holds in all", len(pp), "months")
pp[["pathways", "ratio", "patients", "surplus_pathways", "surplus_share", "pathways_per_patient"]]

patients = ratio × pathways (incl. estimates) holds in all 17 months


,pathways,ratio,patients,surplus_pathways,surplus_share,pathways_per_patient
period,,,,,,
2025-04,7389065,0.8436,"6,233,689.6181","1,155,375.3819",0.1564,1.1853
2025-05,7355950,0.8460,"6,223,356.9298","1,132,593.0702",0.1540,1.1820
2025-06,7367836,0.8458,"6,231,428.3707","1,136,407.6293",0.1542,1.1824
2025-07,7390279,0.8441,"6,237,953.5180","1,152,325.4820",0.1559,1.1847
2025-08,7403612,0.8441,"6,249,489.3438","1,154,122.6562",0.1559,1.1847
2025-09,7388149,0.8441,"6,236,373.6340","1,151,775.3660",0.1559,1.1847
2025-10,7395445,0.8431,"6,235,256.2066","1,160,188.7934",0.1569,1.1861
2025-11,7308928,0.8444,"6,171,719.1647","1,137,208.8353",0.1556,1.1843
2025-12,7290880,0.8454,"6,163,451.4660","1,127,428.5340",0.1546,1.1829


## Step 2 — What "1.1 million people on more than one pathway" can and can't mean

Let *P* be patients, *W* pathways, and *S = W − P* the surplus. Each patient
on *k* pathways contributes *k − 1* to *S*. So if *N₂* is the number of
people on two or more pathways, and *K* the most pathways any one person holds:

> **S / (K − 1) ≤ N₂ ≤ S**

*N₂ = S* only if nobody holds three or more pathways. No published file gives
the distribution of pathways per patient. I checked the WLMDS summary, which
publishes pathway counts but not people. So **S is an upper bound**, and the
true figure is lower by an unknown amount. Below is how far the lower bound
moves for a few illustrative values of *K*. They are illustrations, not estimates.

In [4]:
S = pp.loc[LAST, "surplus_pathways"]
P = pp.loc[LAST, "patients"]
bounds = pd.DataFrame([{"max_pathways_per_person_K": k, "N2_lower_bound": S / (k - 1), "N2_upper_bound": S,
                        "single_pathway_people_at_least": P - S}
                       for k in (2, 3, 4, 5)])
bounds.to_csv(OUT / "multi_pathway_bounds.csv", index=False)
bounds.round(0)

,max_pathways_per_person_K,N2_lower_bound,N2_upper_bound,single_pathway_people_at_least
0,2,"1,127,829.0000","1,127,829.0000","5,114,442.0000"
1,3,"563,915.0000","1,127,829.0000","5,114,442.0000"
2,4,"375,943.0000","1,127,829.0000","5,114,442.0000"
3,5,"281,957.0000","1,127,829.0000","5,114,442.0000"


## Step 3 — How precise is 6.24m?

The press notice states the ratio as **85:100**. The Overview carries more
decimal places. NHS England doesn't state the ratio's uncertainty, so I show
how much the headline patient figure moves with small changes in the ratio:
rounding to the notice's 85:100, and ±0.5 percentage points.

In [5]:
W = pp.loc[LAST, "pathways"]; r = pp.loc[LAST, "ratio"]
sens = pd.DataFrame([
    {"case": "Overview ratio (as published)", "ratio": r},
    {"case": "press-notice ratio 85:100", "ratio": 0.85},
    {"case": "ratio − 0.005", "ratio": r - 0.005},
    {"case": "ratio + 0.005", "ratio": r + 0.005},
]).assign(patients=lambda d: d.ratio * W, surplus=lambda d: W - d.ratio * W)
sens.to_csv(OUT / "ratio_sensitivity.csv", index=False)
sens.round(4)

,case,ratio,patients,surplus
0,Overview ratio (as published),0.8470,"6,242,270.8339","1,127,829.1661"
1,press-notice ratio 85:100,0.8500,"6,264,585.0000","1,105,515.0000"
2,ratio − 0.005,0.8420,"6,205,420.3339","1,164,679.6661"
3,ratio + 0.005,0.8520,"6,279,121.3339","1,090,978.6661"


## Step 4 — When "patients" moves, is it people or pathways?

Because patients = ratio × pathways, any change in the patient figure
splits exactly into a **pathway effect** (the list moved, at a constant
ratio) and a **ratio effect** (the number of people per pathway changed).
I use the midpoint split, which adds up exactly:

> ΔP = r̄ · ΔW + W̄ · Δr

In [6]:
def decompose(a: str, b: str) -> dict:
    ra, rb, wa, wb = pp.loc[a, "ratio"], pp.loc[b, "ratio"], pp.loc[a, "pathways"], pp.loc[b, "pathways"]
    dP = pp.loc[b, "patients"] - pp.loc[a, "patients"]
    pw, rt = (ra + rb) / 2 * (wb - wa), (wa + wb) / 2 * (rb - ra)
    assert abs(pw + rt - dP) < 1
    return {"from": a, "to": b, "patients_change": dP, "pathway_effect": pw, "ratio_effect": rt,
            "pathways_change": wb - wa, "ratio_from": ra, "ratio_to": rb}

decomp = pd.DataFrame([decompose(PREV, LAST), decompose("2025-08", LAST),
                       decompose("2026-03", LAST), decompose(periods[0], LAST)])
decomp.to_csv(OUT / "patients_change_decomposition.csv", index=False)
decomp.round(4)

,from,to,patients_change,pathway_effect,ratio_effect,pathways_change,ratio_from,ratio_to
0,2026-07,2026-08,"35,198.0484","35,444.8010",-246.7525,41848,0.8470,0.8470
1,2025-08,2026-08,"-7,218.5099","-28,335.8355","21,117.3255",-33512,0.8441,0.8470
2,2026-03,2026-08,"223,668.1132","223,607.2095",60.9037,264009,0.8470,0.8470
3,2025-04,2026-08,"8,581.2157","-16,031.2043","24,612.4201",-18965,0.8436,0.8470


## Step 5 — Two collections, one ratio: do they agree on the list?

The ratio comes from the WLMDS. The pathways it is applied to come from the
monthly RTT return. If the two collections disagreed materially on the size
of the list, applying one's ratio to the other's count would be shaky. The
WLMDS summary publishes its own national total for week ending 30 August
2026, both as submitted and with NHS England's estimates for missing trusts.

In [7]:
wl = openpyxl.load_workbook(BRONZE_DIR / "nhse_wlmds_2026-08" / "WLMDS-Summary-to-30-August-2026.xlsx",
                            read_only=True, data_only=True)["National-Time Series"]
wl.reset_dimensions()
wrows = list(wl.iter_rows(values_only=True))
assert str(wrows[13][4]).startswith("Total Waiting List"), wrows[13][4]
we = [r for r in wrows if hasattr(r[1], "strftime") and r[1].strftime("%Y-%m-%d") == "2026-08-30"]
# The first 30-Aug row is as submitted; the second sits under an 'Including estimates' label.
assert len(we) == 2, f"expected two week-ending 30 Aug 2026 rows, found {len(we)}"
lab = next(r[1] for r in wrows if isinstance(r[1], str) and r[1].lower().startswith("including esti"))
wlmds = pd.DataFrame([
    {"basis": "as submitted", "wlmds_30_aug": we[0][4], "rtt_monthly_aug": pp.loc[LAST, "pathways_reported"]},
    {"basis": "with estimates", "wlmds_30_aug": we[1][4], "rtt_monthly_aug": pp.loc[LAST, "pathways"]},
]).assign(diff=lambda d: d.rtt_monthly_aug - d.wlmds_30_aug, diff_pct=lambda d: 100 * (d.rtt_monthly_aug / d.wlmds_30_aug - 1))
print("label on the second row:", lab)
wlmds.to_csv(OUT / "wlmds_vs_rtt.csv", index=False)
wlmds.round(3)

label on the second row: Including estimates for missing data and data quality issues at NHS Acute providers


,basis,wlmds_30_aug,rtt_monthly_aug,diff,diff_pct
0,as submitted,7122072,7158164,36092,0.5070
1,with estimates,7333277,7370100,36823,0.5020


## Step 6 — On my own measure (illustration only)

NHS England applies the ratio to the **headline** pathway count. My RTT
pieces use the reported-provider count. If I want a patient figure on the
same basis as my other figures, applying the same ratio gives the number
below. **This is not an NHS England statistic**, and if I use it I have to
say so. It also assumes the WLMDS ratio holds for the reporting trusts alone.

In [8]:
illus = {"pathways_reported": pp.loc[LAST, "pathways_reported"],
         "ratio": r,
         "patients_illustrative": r * pp.loc[LAST, "pathways_reported"],
         "surplus_illustrative": (1 - r) * pp.loc[LAST, "pathways_reported"]}
pd.Series(illus).to_frame("value").to_csv(OUT / "reported_basis_illustration.csv")
pd.Series(illus).round(4)

pathways_reported       7,158,164.0000
ratio                           0.8470
patients_illustrative   6,062,766.9043
surplus_illustrative    1,095,397.0957
dtype: float64

## Step 7 — Verdict, computed

In [9]:
a = pp.loc[LAST]
print(f"August 2026: {a.pathways:,.0f} pathways (headline basis, rebuilt exactly); "
      f"NHS England estimates {a.patients:,.0f} patients (WLMDS ratio {a.ratio:.4f} × pathways).")
print(f"Surplus pathways: {a.surplus_pathways:,.0f} ({a.surplus_share:.1%} of pathways; {a.pathways_per_patient:.3f} pathways per patient).")
print(f"=> AT MOST {a.surplus_pathways:,.0f} people are on more than one pathway; if anyone holds 3+, fewer. "
      f"'1.1m people on more than one pathway' overstates what the data supports; '1.1m more pathways than patients' is exact.")
print(f"Ratio range Apr25-Aug26: {pp.ratio.min():.4f} to {pp.ratio.max():.4f} "
      f"(surplus {pp.surplus_pathways.min():,.0f} to {pp.surplus_pathways.max():,.0f}).")
d = decomp.set_index(["from", "to"])
for (f, t), row in d.iterrows():
    print(f"patients {f}->{t}: {row.patients_change:+,.0f} = pathway effect {row.pathway_effect:+,.0f} "
          f"+ ratio effect {row.ratio_effect:+,.0f}")
s85 = sens.set_index("case").loc["press-notice ratio 85:100", "patients"]
print(f"Precision: the notice's rounded 85:100 would give {s85:,.0f} patients ({s85 - a.patients:+,.0f}); "
      f"±0.005 on the ratio moves patients by ±{0.005 * a.pathways:,.0f}.")
w2 = wlmds.set_index("basis").loc["with estimates"]
print(f"Collections: monthly RTT {w2.rtt_monthly_aug:,.0f} vs WLMDS 30 Aug {w2.wlmds_30_aug:,.0f} "
      f"({w2.diff_pct:+.2f}%), both with estimates.")
print(f"On my reported-provider measure (illustrative, NOT an NHS England figure): "
      f"{illus['patients_illustrative']:,.0f} patients, {illus['surplus_illustrative']:,.0f} surplus pathways.")

August 2026: 7,370,100 pathways (headline basis, rebuilt exactly); NHS England estimates 6,242,271 patients (WLMDS ratio 0.8470 × pathways).
Surplus pathways: 1,127,829 (15.3% of pathways; 1.181 pathways per patient).
=> AT MOST 1,127,829 people are on more than one pathway; if anyone holds 3+, fewer. '1.1m people on more than one pathway' overstates what the data supports; '1.1m more pathways than patients' is exact.
Ratio range Apr25-Aug26: 0.8431 to 0.8470 (surplus 1,087,488 to 1,160,189).
patients 2026-07->2026-08: +35,198 = pathway effect +35,445 + ratio effect -247
patients 2025-08->2026-08: -7,219 = pathway effect -28,336 + ratio effect +21,117
patients 2026-03->2026-08: +223,668 = pathway effect +223,607 + ratio effect +61
patients 2025-04->2026-08: +8,581 = pathway effect -16,031 + ratio effect +24,612
Precision: the notice's rounded 85:100 would give 6,264,585 patients (+22,314); ±0.005 on the ratio moves patients by ±36,850.
Collections: monthly RTT 7,370,100 vs WLMDS 30 Aug